# Walkthrough — Hugging Face checkpoint to Hailo-10H HEF

This notebook reproduces **end to end** the compilation chain of a
self-compiled LLM for the Hailo-10H, with validation (cosine + inference)
at every step where it is possible.

Quantization recipe: `saitama`/GPU engine, **without** `weight_group_size`,
**without** `bias_correction` or `adaround`, with
`pre_quantization_optimization(ew_add_fusing, policy=disabled)` (the
official Qwen recipe disables it explicitly), and calibration of the RoPE
inputs in **raw integer positions** (no precomputed cos/sin).

Five real incompatibilities between a vanilla HF export and what the 10H
LLM stack expects are baked in along the way — each is summarized at the
step where it lands and evidenced in
[docs/findings/](../docs/findings/index.md):

| # | Fix | Where |
|---|-----|-------|
| 1 | config key `prefill_input_tokens_count` (`_size` is silently ignored → default 96) | step 5 |
| 2 | asymmetric RoPE input widths (K = θ·n_kv_heads, Q = θ·n_heads), duplicate convs deleted | step 5 |
| 3 | mask broadcast semantics (`repeat` AABBCC ≠ `tile` ABCABC) → direct wiring to `input_layer2` | step 5 |
| 4 | explicit `lm_head` matmul (the runtime argmaxes the raw output) | step 2 |
| 5 | last-position slice before `lm_head` → `[1,1,vocab]` NHWC output | steps 2–3 |

## Validation status

| Step | Validation | Status |
|------|------------|--------|
| 1. HF download | cosine + inference | OK |
| 2. matmul-tricks reimplementation (+ lm_head) | cosine (last position) | OK (= 1.000000 vs HF) |
| 3. → ONNX | cosine | OK (= 1.000000) |
| 4. ONNX → HAR (native fp32) | cosine (SDK_NATIVE) | OK (= 1.000000) |
| 5. surgery + resources | cosine (SDK_NATIVE) | OK (= 1.000000 after surgery) |
| 6. KV-cache quantization | — | quantization itself OK; emulator structurally broken on KV-cache graphs (see step 6 note) |
| 7. convolution repair | structural | OK (0 misaligned convs with this recipe) |
| 8. → HEF | on-chip test | prefill exact; base-scope greedy generation coherent; multi-token `__tbt` generation still degraded — [open issue](../docs/findings/open-tbt-cache-read.md) |

## Environment (launch before opening this notebook)

```bash
# Toolchain image built from docker/Dockerfile.amd or Dockerfile.nvidia
docker run -d --name dfc-notebook \
  --device=/dev/kfd --device=/dev/dri --group-add $(getent group render | cut -d: -f3) \
  --group-add $(getent group video | cut -d: -f3) \
  --security-opt seccomp=unconfined --ipc=host \
  -e CUDA_VISIBLE_DEVICES=0 -e NPY_PROMOTION_STATE=legacy \
  -p 8888:8888 \
  -v "$PWD":/repo -v "$PWD/workdir":/workdir \
  -e DFC_WORKDIR=/workdir \
  dfc-amd:5.3.0 \
  jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

Open the forwarded port, execute the cells in order.
Total duration ≈ **15 min** on GPU (quantization + compile), versus hours
on CPU.

## 0. Common setup (run once)

In [ ]:
import os

# MUST be set before numpy is imported anywhere in this kernel:
# NumPy >= 2 scalar promotion breaks create_hw_params() in the DFC.
os.environ.setdefault("NPY_PROMOTION_STATE", "legacy")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USER", "root")

import importlib
import inspect
import json
import math
import pkgutil

import keras
import hailo_model_optimization.acceleras as _acceleras_pkg

# Required so the SDK can deserialize the internal (Keras) layers it
# writes/reads inside quantized HARs.
_registered = 0
for _finder, _modname, _ in pkgutil.walk_packages(
    _acceleras_pkg.__path__, prefix="hailo_model_optimization.acceleras."
):
    try:
        _mod = importlib.import_module(_modname)
    except Exception:
        continue
    for _attr_name, _attr in inspect.getmembers(_mod, inspect.isclass):
        if (
            _attr.__module__ == _modname
            and issubclass(_attr, keras.layers.Layer)
            and getattr(_attr, "_keras_api_names", None) is None
        ):
            keras.saving.register_keras_serializable()(_attr)
            _registered += 1
print(f"{_registered} Keras/acceleras layer classes registered")

import numpy as np
import onnxruntime as ort
import torch
from hailo_sdk_client import ClientRunner
from hailo_sdk_client.exposed_definitions import Dims, InferenceContext, DistributionStrategy
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Model under compilation (LLaMA2-style with GQA) -----------------------
MODEL_ID = "Mxode/TinyStories-LLaMA2-25M-256h-4l-GQA"
HIDDEN = 256
NHEAD = 16
NKVHEAD = 8
NREP = NHEAD // NKVHEAD
HD = HIDDEN // NHEAD          # head dimension (16)
Q_WIDTH = NHEAD * HD
NLAYERS = 4
ROPE_THETA = 10000.0
RMS_EPS = 1e-6
SEQ = 24                      # calibration/parse length of the base scope
PREFILL_SIZE = 16             # __prefill scope length
CACHE_SIZE = 24               # total KV-cache size -- MUST equal SEQ here
VOCAB = 32000
CALIBSET_SIZE = 32
NET_SCOPE = "ts25mpipe"
BOS_TOKEN_ID = 1
EOS_TOKEN_ID = 2
PAD_TOKEN_ID = EOS_TOKEN_ID   # this tokenizer has no dedicated pad token

# --- Artifacts -------------------------------------------------------------
WORKDIR = os.environ.get("DFC_WORKDIR", os.path.join(os.getcwd(), "workdir"))
os.makedirs(WORKDIR, exist_ok=True)
ONNX_PATH = os.path.join(WORKDIR, "model.onnx")
HAR_PATH = os.path.join(WORKDIR, "parsed.har")
HAR_SURGERY_PATH = os.path.join(WORKDIR, "surgery.har")
HAR_RESOURCES_PATH = os.path.join(WORKDIR, "resources.har")
Q_HAR_PATH = os.path.join(WORKDIR, "quantized.har")
CONVFIXED_HAR_PATH = os.path.join(WORKDIR, "convfixed.har")
HEF_PATH = os.path.join(WORKDIR, "model.hef")
WTE_PATH = os.path.join(WORKDIR, "wte.npy")
HF_REFS_PATH = os.path.join(WORKDIR, "hf_reference.npz")
TOKENIZER_JSON_PATH = os.path.join(WORKDIR, "tokenizer.json")


def cosine(a, b):
    """Flat cosine similarity in float64 -- the project-wide fidelity metric."""
    a = a.flatten().astype(np.float64)
    b = b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


print("workdir:", WORKDIR)

## Step 1 — Load the HF checkpoint, build the float reference

Loads the model + tokenizer, computes the PyTorch reference logits (`hf_out`)
used by every later cosine validation. **The test prompt is exactly
`SEQ` = 24 real tokens, without padding** — required since fix #5: the graph
is sliced at the last position before `lm_head`, so comparing against a pad
position would be meaningless.

Also persists `wte.npy`, `hf_reference.npz` and `tokenizer.json` into
`WORKDIR` — the embedding table and reference tensors reused by the
[diagnostics notebook](diagnostics.ipynb) and by step 5's embedded
resources.

In [ ]:
print(f"==> loading {MODEL_ID}")
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
wte = hf_model.model.embed_tokens.weight.detach().numpy().astype(np.float32)
lm_head_w = hf_model.lm_head.weight.detach().numpy().astype(np.float32)
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

assert wte.shape == (VOCAB, HIDDEN)
assert lm_head_w.shape == (VOCAB, HIDDEN)
assert not (hf_model.lm_head.weight is hf_model.model.embed_tokens.weight), "unexpected tie_word_embeddings"

PROMPT = "Once upon a time there was a little girl"
LONG_PROMPT = (PROMPT + " ") * 6  # long enough to cut exactly SEQ tokens, no padding
ids_long = tokenizer(LONG_PROMPT, return_tensors="pt")["input_ids"]
ids = ids_long[:, :SEQ]
S = ids.shape[1]
assert S == SEQ, f"S={S} != SEQ={SEQ}, prompt too short"

with torch.no_grad():
    hf_out = hf_model(ids).logits.numpy()  # (1, SEQ, VOCAB)

token_ids_full = ids.numpy().astype(np.int64)
token_embeds_full = wte[token_ids_full].astype(np.float32)

np.save(WTE_PATH, wte)
np.savez(
    HF_REFS_PATH,
    token_ids=token_ids_full,
    token_embeds=token_embeds_full,
    hf_logits_last=hf_out[:, -1:, :],
    wte=wte,
    pad_id=np.array(pad_id),
)
tokenizer.save_pretrained(WORKDIR)
assert os.path.exists(TOKENIZER_JSON_PATH), "fast tokenizer expected (tokenizer.json)"

print(f"HF logits: {hf_out.shape} -- last position = the prediction compared everywhere below")
print("[OK] step 1 validated")

## Step 2 — Matmul-tricks reimplementation (+ lm_head, last-position slice)

Reimplements the HF model using only simple operations (matmul/add/mul) that
export to ONNX as-is. Native torch RoPE and GQA `repeat_kv` are expressed as
multiplications by constant permutation/tiling matrices. This matmul-tricks
pattern is what the OFFICIAL Hailo/Qwen flow also does internally
(`matmul(groups=..., input_tiles=...)`) — not a project-specific hack: the
native path (`repeat_interleave`/`Expand`) breaks the DFC parser
(`UnsupportedShuffleLayerError`).

**Fixes #4 + #5**: the graph now ends with an `lm_head` matmul (weights
SEPARATE from the embedding, 256→32000) applied ONLY to the last sequence
position (`x[:, -1:, :]`). The two fixes are inseparable: without `lm_head`,
`genai` samples over the cache state; without the slice, the monolithic
`lm_head` (256×32000) cannot be placed on the chip.

In [ ]:
theta_base = 1.0 / (ROPE_THETA ** (np.arange(0, HD, 2, dtype=np.float64) / HD))
theta_np = np.concatenate([theta_base, theta_base]).astype(np.float32)


def rms_norm(x, weight, eps=RMS_EPS):
    variance = x.pow(2).mean(-1, keepdim=True)
    x = x * torch.rsqrt(variance + eps)
    return x * weight


def make_rotate_half_matrix(hd, n_heads):
    perm = torch.zeros(hd, hd)
    for i in range(hd // 2):
        perm[i, i + hd // 2] = 1.0
        perm[i + hd // 2, i] = -1.0
    return torch.block_diag(*([perm] * n_heads))


ROTATE_HALF_Q = make_rotate_half_matrix(HD, NHEAD)
ROTATE_HALF_K = make_rotate_half_matrix(HD, NKVHEAD)


def rotate_half(x, matrix):
    return x @ matrix


def make_tile_matrix(hd, n_heads):
    m = torch.zeros(hd, hd * n_heads)
    for h in range(n_heads):
        for i in range(hd):
            m[i, h * hd + i] = 1.0
    return m


TILE_Q = make_tile_matrix(HD, NHEAD)
TILE_K = make_tile_matrix(HD, NKVHEAD)


def tile_to_width(x, matrix):
    return x @ matrix


def make_repeat_kv_matrix(hd, n_kv_heads, n_rep):
    kv_width = hd * n_kv_heads
    q_width = hd * n_kv_heads * n_rep
    m = torch.zeros(kv_width, q_width)
    for kv in range(n_kv_heads):
        for r in range(n_rep):
            out_head = kv * n_rep + r
            for d in range(hd):
                m[kv * hd + d, out_head * hd + d] = 1.0
    return m


REPEAT_KV_MATRIX = make_repeat_kv_matrix(HD, NKVHEAD, NREP)


def repeat_kv_flat(x):
    return x @ REPEAT_KV_MATRIX


class GQALayer(torch.nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.Wq = torch.nn.Parameter(layer.self_attn.q_proj.weight.detach().T.clone())
        self.Wk = torch.nn.Parameter(layer.self_attn.k_proj.weight.detach().T.clone())
        self.Wv = torch.nn.Parameter(layer.self_attn.v_proj.weight.detach().T.clone())
        self.Wo = torch.nn.Parameter(layer.self_attn.o_proj.weight.detach().T.clone())
        self.ln1_w = torch.nn.Parameter(layer.input_layernorm.weight.detach().clone())
        self.ln2_w = torch.nn.Parameter(layer.post_attention_layernorm.weight.detach().clone())
        self.Wgate = torch.nn.Parameter(layer.mlp.gate_proj.weight.detach().T.clone())
        self.Wup = torch.nn.Parameter(layer.mlp.up_proj.weight.detach().T.clone())
        self.Wdown = torch.nn.Parameter(layer.mlp.down_proj.weight.detach().T.clone())

    def forward(self, x, attention_mask_tiled, k_cos_t, q_cos_t, k_sin_t, q_sin_t):
        residual = x
        h = rms_norm(x, self.ln1_w)
        B_, S_, _ = h.shape
        q = h @ self.Wq
        k = h @ self.Wk
        v = h @ self.Wv
        q = q * q_cos_t + rotate_half(q, ROTATE_HALF_Q) * q_sin_t
        k = k * k_cos_t + rotate_half(k, ROTATE_HALF_K) * k_sin_t
        k = repeat_kv_flat(k)
        v = repeat_kv_flat(v)
        q = q.view(B_, S_, NHEAD, HD).transpose(1, 2)
        k = k.view(B_, S_, NHEAD, HD).transpose(1, 2)
        v = v.view(B_, S_, NHEAD, HD).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(HD)
        mask_bias = attention_mask_tiled[:, :, :, :S_]
        scores = scores + mask_bias
        probs = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).reshape(B_, S_, Q_WIDTH)
        attn_out = attn_out @ self.Wo
        x = residual + attn_out
        residual = x
        h = rms_norm(x, self.ln2_w)
        gate = torch.nn.functional.silu(h @ self.Wgate)
        up = h @ self.Wup
        h = (gate * up) @ self.Wdown
        x = residual + h
        return x


class ExportableModelWithHead(torch.nn.Module):
    """GQA backbone + final lm_head matmul (256->32000, weights SEPARATE from
    the embedding), applied ONLY to the last position (fixes #4 + #5 -- genai
    never predicts more than ONE next token, from the last context position)."""

    def __init__(self, hf_model):
        super().__init__()
        self.layers = torch.nn.ModuleList([GQALayer(hf_model.model.layers[i]) for i in range(NLAYERS)])
        self.norm_w = torch.nn.Parameter(hf_model.model.norm.weight.detach().clone())
        self.Wlm = torch.nn.Parameter(hf_model.lm_head.weight.detach().T.clone())  # (256, 32000)

    def forward(self, token_embeds, attention_mask_tiled, pe_k_cos, pe_q_cos, pe_k_sin, pe_q_sin):
        x = token_embeds
        k_cos_t = tile_to_width(pe_k_cos, TILE_K)
        q_cos_t = tile_to_width(pe_q_cos, TILE_Q)
        k_sin_t = tile_to_width(pe_k_sin, TILE_K)
        q_sin_t = tile_to_width(pe_q_sin, TILE_Q)
        for layer in self.layers:
            x = layer(x, attention_mask_tiled, k_cos_t, q_cos_t, k_sin_t, q_sin_t)
        x = x.reshape(x.shape[0], x.shape[1], HIDDEN)
        x = rms_norm(x, self.norm_w)
        x_last = x[:, -1:, :]  # fix #5: one single position, not the whole sequence
        logits = x_last @ self.Wlm
        return logits


print("==> building exportable reimplementation (with lm_head + last-position slice)")
wrapped = ExportableModelWithHead(hf_model).eval()

pos_full = np.arange(SEQ)
angles = np.outer(pos_full, theta_np)
cos_full = torch.tensor(np.cos(angles), dtype=torch.float32).unsqueeze(0)
sin_full = torch.tensor(np.sin(angles), dtype=torch.float32).unsqueeze(0)

causal_bias = torch.triu(torch.full((SEQ, SEQ), -100.0), diagonal=1)
attention_mask_tiled = causal_bias.view(1, 1, SEQ, SEQ).repeat(1, 1, 1, NHEAD)

with torch.no_grad():
    logits_wrapped = wrapped(
        torch.tensor(token_embeds_full), attention_mask_tiled, cos_full, cos_full, sin_full, sin_full
    ).numpy()  # (1, 1, VOCAB)

cos_wrapped_vs_hf = cosine(hf_out[:, -1:, :], logits_wrapped)
print(f"cosine(HF last position, PyTorch reimplementation): {cos_wrapped_vs_hf:.6f}")
assert cos_wrapped_vs_hf > 0.999, "reimplementation does not match HF"
print("[OK] step 2 validated -- faithful reimplementation (lm_head + slice included)")

## Step 3 — Reimplementation → ONNX

Exports the PyTorch graph to ONNX (opset 17, `dynamo=False` — the legacy
TorchScript exporter, a deliberate choice: predictable tracing, already
validated on this project; `do_constant_folding=False` keeps the
constant-matrix structure visible to the parser). Single `logits` output
with shape `[1,1,32000]` — made placeable by fix #5.

In [ ]:
print("==> ONNX export + onnxruntime inference")
torch.onnx.export(
    wrapped,
    (torch.randn(1, SEQ, HIDDEN), attention_mask_tiled, cos_full, cos_full, sin_full, sin_full),
    ONNX_PATH,
    input_names=["inputs_embeds", "attention_mask", "pe_k_cos", "pe_q_cos", "pe_k_sin", "pe_q_sin"],
    output_names=["logits"],
    opset_version=17,
    do_constant_folding=False,
    dynamo=False,
)
print(f"ONNX exported: {ONNX_PATH} ({os.path.getsize(ONNX_PATH)} bytes)")

sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
onnx_inputs_ref = {
    "inputs_embeds": token_embeds_full.astype(np.float32),
    "attention_mask": attention_mask_tiled.numpy().astype(np.float32),
    "pe_k_cos": cos_full.numpy().astype(np.float32),
    "pe_q_cos": cos_full.numpy().astype(np.float32),
    "pe_k_sin": sin_full.numpy().astype(np.float32),
    "pe_q_sin": sin_full.numpy().astype(np.float32),
}
onnx_logits = sess.run(["logits"], onnx_inputs_ref)[0]
cos_onnx_vs_hf = cosine(hf_out[:, -1:, :], onnx_logits)
print(f"cosine(HF last position, ONNX/onnxruntime): {cos_onnx_vs_hf:.6f}")
assert cos_onnx_vs_hf > 0.999, "ONNX does not match HF"
print("[OK] step 3 validated -- faithful ONNX")

## Step 4 — ONNX → HAR (native fp32 parse)

Parses the ONNX graph into a native Hailo HAR. At this point
`input_layer3-6` (RoPE) are still declared at uniform width (16, untiled)
and `input_layer2` (mask) still goes through an intermediate `slice` before
`ew_add` — fixes #2/#3 are not applied yet, but the graph remains
mathematically correct in native FP32 (`SDK_NATIVE`, no quantization)
because those two bugs only affect REAL runtime behavior on genai/the chip,
not the DFC software simulation itself. Checked here as a sanity gate
before step 5's surgery.

The declared input shapes/formats are contractual and matter:

In [ ]:
print("==> parsing ONNX -> HAR")
runner = ClientRunner(hw_arch="hailo10h")
runner.translate_onnx_model(
    ONNX_PATH,
    NET_SCOPE,
    disable_onnx_simplifier=True,
    net_input_shapes={
        "inputs_embeds": [1, SEQ, HIDDEN],
        "attention_mask": [1, 1, SEQ, NHEAD * SEQ],
        "pe_k_cos": [1, SEQ, HD],
        "pe_q_cos": [1, SEQ, HD],
        "pe_k_sin": [1, SEQ, HD],
        "pe_q_sin": [1, SEQ, HD],
    },
    net_input_format={
        "inputs_embeds": [Dims.BATCH, Dims.WIDTH, Dims.CHANNELS],
        "attention_mask": [Dims.BATCH, Dims.HEIGHT, Dims.WIDTH, Dims.CHANNELS],
        "pe_k_cos": [Dims.BATCH, Dims.WIDTH, Dims.CHANNELS],
        "pe_q_cos": [Dims.BATCH, Dims.WIDTH, Dims.CHANNELS],
        "pe_k_sin": [Dims.BATCH, Dims.WIDTH, Dims.CHANNELS],
        "pe_q_sin": [Dims.BATCH, Dims.WIDTH, Dims.CHANNELS],
    },
)
runner.save_har(HAR_PATH)
print(f"[OK] parse+save succeeded -> {HAR_PATH}")

causal_bias_np = np.triu(np.full((SEQ, SEQ), -100.0, dtype=np.float32), k=1)
attn_mask_np = np.tile(causal_bias_np[np.newaxis, np.newaxis, :, :], (1, 1, 1, NHEAD)).astype(np.float32)
calib_native = {
    f"{NET_SCOPE}/input_layer1": token_embeds_full[:, np.newaxis, :, :].astype(np.float32),
    f"{NET_SCOPE}/input_layer2": attn_mask_np,
    f"{NET_SCOPE}/input_layer3": cos_full.numpy()[:, np.newaxis, :, :].astype(np.float32),
    f"{NET_SCOPE}/input_layer4": cos_full.numpy()[:, np.newaxis, :, :].astype(np.float32),
    f"{NET_SCOPE}/input_layer5": sin_full.numpy()[:, np.newaxis, :, :].astype(np.float32),
    f"{NET_SCOPE}/input_layer6": sin_full.numpy()[:, np.newaxis, :, :].astype(np.float32),
}
with runner.infer_context(InferenceContext.SDK_NATIVE, gpu_policy=DistributionStrategy.SINGLE) as ctx:
    out = runner.infer(ctx, dataset=calib_native, data_type="np_array", batch_size=1)
native_logits = np.array(out[0] if isinstance(out, list) else out).reshape(1, 1, -1)
cos_native_vs_hf = cosine(hf_out[:, -1:, :], native_logits)
print(f"cosine(HF last position, HAR/SDK_NATIVE, BEFORE surgery): {cos_native_vs_hf:.6f}")
print("[OK] step 4 validated")

## Step 5 — Structural surgery (RoPE + mask) + external resources

**Fix #2 (RoPE)**: `input_layer3-6` are declared at uniform width (16)
while `genai` actually writes, at runtime, buffers tiled asymmetrically
(`K = θ_size × num_key_value_heads = 128`, `Q = θ_size × num_attention_heads
= 256`) — verified against the official Qwen `.native.hn`. A duplication
conv (`conv1-4`, period-16 pattern, EXACTLY genai's `tile_along_last_axis`
mechanism but executed TWICE on-chip) did this tiling downstream — deleted;
`input_layer3-6` redeclared directly at the final tiled widths, reconnected
to their `ew_mult*` consumers.

**Fix #3 (mask)**: `input_layer2` (already ×16-tiled by construction
host-side, exactly like `genai::prepare_attention_mask_input`) went through
a `slice` (re-cut to 24) then an `ew_add` with `input_repeats` (wrong
`tf.repeat` semantics, AABBCC instead of ABCABC) to widen back to 384 before
being added to attention scores — deleted; `ew_add*` reconnected DIRECTLY to
`input_layer2` (already at the right width, no broadcast needed).

Then re-attaches the four external resources (on-chip embedding + RoPE
theta + tokenizer + hailo-config) onto the fixed HAR.

In [ ]:
import tarfile
import tempfile

print("=== surgery 1/2: RoPE (delete redundant duplication) ===")
ROPE_LAYERS = [
    (f"{NET_SCOPE}/input_layer3", f"{NET_SCOPE}/conv1", NKVHEAD),  # pe_k_cos
    (f"{NET_SCOPE}/input_layer4", f"{NET_SCOPE}/conv2", NHEAD),    # pe_q_cos
    (f"{NET_SCOPE}/input_layer5", f"{NET_SCOPE}/conv3", NKVHEAD),  # pe_k_sin
    (f"{NET_SCOPE}/input_layer6", f"{NET_SCOPE}/conv4", NHEAD),    # pe_q_sin
]
MASK_LAYERS = [(1, "ew_add3"), (2, "ew_add8"), (3, "ew_add13"), (4, "ew_add18")]

with tempfile.TemporaryDirectory() as d:
    with tarfile.open(HAR_PATH) as t:
        t.extractall(d)
    hn_files = [f for f in os.listdir(d) if f.endswith(".hn") and not f.endswith((".fp.hn", ".native.hn"))]
    hn_path = os.path.join(d, hn_files[0])
    hn = json.load(open(hn_path))
    layers = hn["layers"]

    for input_name, conv_name, groups in ROPE_LAYERS:
        conv = layers[conv_name]
        consumers = conv["output"]
        new_width = conv["output_shapes"][0][-1]
        assert new_width == HD * groups, f"{conv_name}: {new_width} != {HD * groups}"
        inp = layers[input_name]
        old_shape = inp["output_shapes"][0]
        new_shape = old_shape[:-1] + [new_width]
        inp["input_shapes"] = [new_shape]
        inp["output_shapes"] = [new_shape]
        inp["output"] = list(consumers)
        for cname in consumers:
            c = layers[cname]
            c["input"] = [input_name if x == conv_name else x for x in c["input"]]
            c["input_shapes"] = [new_shape if x == input_name else s for x, s in zip(c["input"], c["input_shapes"])]
        del layers[conv_name]
        print(f"  {input_name}: {old_shape} -> {new_shape} ; {conv_name} deleted")

    print("=== surgery 2/2: mask (rewire straight to input_layer2) ===")
    il2 = layers[f"{NET_SCOPE}/input_layer2"]
    for i, ewname in MASK_LAYERS:
        sname = f"{NET_SCOPE}/slice{i}"
        ewfull = f"{NET_SCOPE}/{ewname}"
        ew = layers[ewfull]
        ew["input"] = [(f"{NET_SCOPE}/input_layer2" if x == sname else x) for x in ew["input"]]
        ew["input_shapes"] = [[-1, 1, SEQ, NHEAD * SEQ], [-1, 1, SEQ, NHEAD * SEQ]]
        ew["params"]["input_repeats"] = [[1, 1, 1], [1, 1, 1]]
        if "input_tiles" in ew["params"]:
            del ew["params"]["input_tiles"]
        il2["output"] = [(ewfull if x == sname else x) for x in il2["output"]]
        del layers[sname]
        print(f"  {ewname}: rewired to input_layer2 directly ; slice{i} deleted")

    json.dump(hn, open(hn_path, "w"))
    with tarfile.open(HAR_SURGERY_PATH, "w") as t:
        for f in os.listdir(d):
            t.add(os.path.join(d, f), arcname=f)
print(f"surgery HAR saved: {HAR_SURGERY_PATH}")

print("\n=== SDK reload check + post-surgery cosine ===")
runner = ClientRunner(har=HAR_SURGERY_PATH)
for input_name, conv_name, groups in ROPE_LAYERS:
    assert conv_name not in runner.get_hn_dict()["layers"]
for i, ewname in MASK_LAYERS:
    assert f"{NET_SCOPE}/slice{i}" not in runner.get_hn_dict()["layers"]

# cos/sin PRE-TILED the way genai really computes them at runtime
# (host-side, never on-chip -- DFC's conversion_type=cos/sin mechanism is
# just a software emulation used for calibration).
def tile_groups(base, groups):
    return np.tile(base, (1, groups)).reshape(1, 1, SEQ, groups * HD).astype(np.float32)

cos_base_np = np.cos(angles).astype(np.float32)
sin_base_np = np.sin(angles).astype(np.float32)
calib_surgery = {
    f"{NET_SCOPE}/input_layer1": token_embeds_full[:, np.newaxis, :, :].astype(np.float32),
    f"{NET_SCOPE}/input_layer2": attn_mask_np,
    f"{NET_SCOPE}/input_layer3": tile_groups(cos_base_np, NKVHEAD),
    f"{NET_SCOPE}/input_layer4": tile_groups(cos_base_np, NHEAD),
    f"{NET_SCOPE}/input_layer5": tile_groups(sin_base_np, NKVHEAD),
    f"{NET_SCOPE}/input_layer6": tile_groups(sin_base_np, NHEAD),
}
with runner.infer_context(InferenceContext.SDK_NATIVE, gpu_policy=DistributionStrategy.SINGLE) as ctx:
    out = runner.infer(ctx, dataset=calib_surgery, data_type="np_array", batch_size=1)
surgery_logits = np.array(out[0] if isinstance(out, list) else out).reshape(1, 1, -1)
cos_surgery_vs_hf = cosine(hf_out[:, -1:, :], surgery_logits)
print(f"cosine(HF last position, HAR/SDK_NATIVE, AFTER surgery): {cos_surgery_vs_hf:.6f}")
assert cos_surgery_vs_hf > 0.999, "surgery broke model fidelity"

print("\n=== re-attach external resources ===")
theta_tile = np.concatenate([theta_base, theta_base]).astype(np.float32)
chat_template = (
    "{% for message in messages %}"
    "{% for item in message['content'] %}"
    "{{ item['text'] }}"
    "{% endfor %}"
    "{% if not loop.last %}\n{% endif %}"
    "{% endfor %}"
)
hailo_config = {
    "model_name": NET_SCOPE,
    "stop_token_id": [EOS_TOKEN_ID],
    "eos_token_id": EOS_TOKEN_ID,
    "default_generation_params": {
        "max_new_tokens": 64, "temperature": 0.7, "top_p": 0.9,
        "top_k": 50, "repetition_penalty": 1.1, "do_sample": True,
    },
    "chat_template": chat_template,
    "pre_process_params": {
        "num_attention_heads": NHEAD, "num_key_value_heads": NKVHEAD,
        "kv_cache_size": CACHE_SIZE, "prefill_input_tokens_count": PREFILL_SIZE,
    },
    "input_layers_names_suffixes": {
        "embeddings": "input_layer1",
        "attention_mask": "input_layer2",
        "pe_k_cos": "input_layer3",
        "pe_q_cos": "input_layer4",
        "pe_k_sin": "input_layer5",
        "pe_q_sin": "input_layer6",
    },
}
hailo_config_path = os.path.join(WORKDIR, "hailo-config.json")
with open(hailo_config_path, "w") as f:
    json.dump(hailo_config, f, indent=2)

runner.add_external_resources({
    "input_layers_mapping": {
        f"{NET_SCOPE}/input_layer1": "embedding",
        f"{NET_SCOPE}/input_layer3": "cos",
        f"{NET_SCOPE}/input_layer4": "cos",
        f"{NET_SCOPE}/input_layer5": "sin",
        f"{NET_SCOPE}/input_layer6": "sin",
    },
    "weights": {
        f"{NET_SCOPE}/input_layer1": {"embed": wte},
        f"{NET_SCOPE}/input_layer3": {"theta": theta_tile, "tile": np.array([1, 1, NKVHEAD], dtype=np.int32), "factor": np.array([1.0], dtype=np.float32)},
        f"{NET_SCOPE}/input_layer4": {"theta": theta_tile, "tile": np.array([1, 1, NHEAD], dtype=np.int32), "factor": np.array([1.0], dtype=np.float32)},
        f"{NET_SCOPE}/input_layer5": {"theta": theta_tile, "tile": np.array([1, 1, NKVHEAD], dtype=np.int32), "factor": np.array([1.0], dtype=np.float32)},
        f"{NET_SCOPE}/input_layer6": {"theta": theta_tile, "tile": np.array([1, 1, NHEAD], dtype=np.int32), "factor": np.array([1.0], dtype=np.float32)},
    },
})
runner.add_external_file("tokenizer.json", TOKENIZER_JSON_PATH)
runner.add_external_file("hailo-config.json", hailo_config_path)
runner.save_har(HAR_RESOURCES_PATH)
print(f"final HAR saved: {HAR_RESOURCES_PATH}")
print("[OK] step 5 validated")

## Step 6 — HAR → quantized HAR (KV-cache quantization, GPU recipe)

The recipe comes from a differential comparison against the official Qwen2
`.alls`: our earlier runs had `pre_quantization_optimization(ew_add_fusing)`
active by default (Qwen explicitly disables it) and added
`bias_correction`/`adaround` that Qwen never uses. **Disabling
`ew_add_fusing` AND dropping `bias_correction`/`adaround`** produced — for
the first time on this project — an EXACTLY correct argmax on real silicon.
WITHOUT `weight_group_size` (incompatible with `saitama`, confirmed SDK bug
in `bias_accumulator.py`). RoPE calibration inputs receive **raw integer
positions** (`(batch, seq)`, NOT precomputed cos/sin — DFC's internal
`conversion_type=cos/sin` mechanism recomputes them for calibration; it is
a software emulation never executed as-is on the chip).

This step also duplicates the graph into the `__prefill`/`__tbt` network
groups via `set_kv_cache_global_params()` — the entry point of the LLM flow.

In [ ]:
print("==> discovering compressible convs (exclusion by sparsity/width)")
runner = ClientRunner(har=HAR_RESOURCES_PATH)
hn = runner.get_hn()
all_convs = [(n, l) for n, l in hn["layers"].items() if l.get("type") == "conv"]
params = runner.get_params()

def _sparsity(name):
    inner = params.get(name)
    if inner is None:
        return 0.0
    kernel = inner.get("kernel:0")
    if kernel is None:
        return 0.0
    return float(np.mean(np.asarray(kernel) == 0))

def _min_in_width(l):
    shapes = l.get("input_shapes") or []
    return min((s[-1] for s in shapes), default=10**9)

NARROW_THRESHOLD = HD
SPARSITY_THRESHOLD = 0.9
def _exclude(n, l):
    return _sparsity(n) > SPARSITY_THRESHOLD or _min_in_width(l) <= NARROW_THRESHOLD

conv_names = sorted(n for n, l in all_convs if not _exclude(n, l))
excluded = sorted(n for n, l in all_convs if _exclude(n, l))
print(f"  {len(all_convs)} convs found, {len(conv_names)} compressed to INT4, {len(excluded)} excluded")
conv_list_str = ", ".join(conv_names)

model_script = f"""
pre_quantization_optimization(ew_add_fusing, policy=disabled)
set_kv_cache_global_params({PREFILL_SIZE}, {CACHE_SIZE})
model_optimization_config(globals, multiproc_policy=disabled)
model_optimization_config(calibration, batch_size=1, calibset_size={CALIBSET_SIZE}, use_saitama=True, device=cuda)
model_optimization_flavor(compression_level=4, optimization_level=0)
quantization_param([{NET_SCOPE}/input_layer1], precision_mode=a16_w16)
quantization_param([{conv_list_str}], precision_mode=a8_w4)
"""
print("=== model script (convs omitted) ===")
print(model_script[:400], "...")

print("==> building calibration set")
sentence_pool = [
    "Once upon a time there was a little girl named Lily who loved to play in the garden every day.",
    "The old wizard walked slowly through the forest, looking for herbs to make his special potion.",
    "Tom and his dog Max went for a walk in the park and found a shiny red ball under a tree.",
    "The little mouse was scared of the big cat, so it hid inside a small hole in the wall.",
    "Every morning the farmer fed his chickens and collected fresh eggs for breakfast.",
    "The children built a sandcastle on the beach and watched the waves crash against the shore.",
    "A kind old man lived in a small cottage at the edge of the village near the river.",
    "The bright yellow sun rose over the mountains as the birds began to sing their morning song.",
]
rng = np.random.default_rng(0)
token_ids_list = []
for _ in range(CALIBSET_SIZE):
    order = rng.permutation(len(sentence_pool))
    calib_ids = []
    for idx in order:
        calib_ids.extend(tokenizer(sentence_pool[idx])["input_ids"])
        if len(calib_ids) >= SEQ:
            break
    calib_ids = (calib_ids + [pad_id] * SEQ)[:SEQ]
    token_ids_list.append(np.array(calib_ids, dtype=np.int64))
calib_token_ids = np.stack(token_ids_list, axis=0)
calib_embeds = wte[calib_token_ids][:, np.newaxis, :, :].astype(np.float32)

calib_causal_bias = np.triu(np.full((SEQ, SEQ), -100.0, dtype=np.float32), k=1)
calib_attn_mask = np.tile(calib_causal_bias[np.newaxis, np.newaxis, :, :], (CALIBSET_SIZE, 1, 1, NHEAD)).astype(np.float32)
calib_raw_positions = np.tile(np.arange(SEQ)[np.newaxis, :], (CALIBSET_SIZE, 1)).astype(np.float32)

calib_data = {
    f"{NET_SCOPE}/input_layer1": calib_embeds,
    f"{NET_SCOPE}/input_layer2": calib_attn_mask,
    f"{NET_SCOPE}/input_layer3": calib_raw_positions,
    f"{NET_SCOPE}/input_layer4": calib_raw_positions,
    f"{NET_SCOPE}/input_layer5": calib_raw_positions,
    f"{NET_SCOPE}/input_layer6": calib_raw_positions,
}
print({k: v.shape for k, v in calib_data.items()})

runner.load_model_script(model_script)
runner.optimize(calib_data)
runner.save_har(Q_HAR_PATH)
print(f"quantized HAR saved: {Q_HAR_PATH}")
print("[OK] step 6 -- GPU quantization done (order of a minute on GPU)")
print("(no cosine available here -- the SDK_QUANTIZED emulator is structurally broken on KV-cache graphs)")

> **Why no cosine gate past this point?** The `SDK_QUANTIZED` emulator is
> structurally broken on KV-cache graphs (two stacked SDK bugs, confirmed by
> source reading, unrelated to the five fixes):
>
> 1. `Cache._get_prefill_size` (`acceleras/utils/cache.py`) raises `TypeError`
>    when `lora_adapter_name=None` (universal SDK bug — reproduced on the
>    official Qwen HAR too).
> 2. Once patched, a structural shape inconsistency appears in
>    `__prefill/matmul1` (256 vs 384), independent of the data provided.
>
> Consequence: no direct cosine can be measured on the quantized KV-cache
> HAR — the only reliable judge for these stages is real hardware.

## Step 7 — Convolution repair safety net

`set_kv_cache_global_params` duplicates the graph into `__prefill`/`__tbt`
scopes, which used to misalign `input`/`input_shapes` on residual-fused
convs when `ew_add_fusing` was active. **With the current recipe (step 6),
0 convs come out misaligned** — strong hint that `ew_add_fusing` was the
root cause. Kept as a robustness net: it reorders `input` lists to match
`input_shapes` by width whenever needed.

In [ ]:
import glob


def fix_duplicated_conv_inputs(har_in: str, har_out: str) -> int:
    with tempfile.TemporaryDirectory() as d:
        with tarfile.open(har_in) as t:
            t.extractall(d)
        hn_path = [p for p in glob.glob(os.path.join(d, "*.hn")) if not p.endswith((".fp.hn", ".native.hn"))][0]
        hn = json.load(open(hn_path))
        layers = hn["layers"]

        def feat(name):
            L = layers.get(name)
            return L["output_shapes"][0][-1] if L and L.get("output_shapes") else None

        fixed = 0
        for L in layers.values():
            if L.get("type") != "conv":
                continue
            ins, shp = L.get("input"), L.get("input_shapes")
            if not ins or not shp or len(ins) < 2:
                continue
            want = [s[-1] for s in shp]
            if [feat(n) for n in ins] == want:
                continue
            pool, newins, ok = list(ins), [], True
            for w in want:
                cand = next((n for n in pool if feat(n) == w), None)
                if cand is None:
                    ok = False
                    break
                newins.append(cand)
                pool.remove(cand)
            if ok and newins != ins:
                print(f"  {L.get('name', '?')}: {ins} -> {newins}")
                L["input"] = newins
                fixed += 1
        json.dump(hn, open(hn_path, "w"))
        with tarfile.open(har_out, "w") as t:
            for f in os.listdir(d):
                t.add(os.path.join(d, f), arcname=f)
    return fixed


print("scanning and fixing misaligned convs...")
n_fixed = fix_duplicated_conv_inputs(Q_HAR_PATH, CONVFIXED_HAR_PATH)
print(f"convs repaired: {n_fixed} (expected 0 with the standard recipe)")
print(f"repaired HAR saved: {CONVFIXED_HAR_PATH}")
print("[OK] step 7 validated")

## Step 8 — quantized HAR → HEF (compile)

Compiles the quantized HAR into a HEF with the two network groups declared
by their exact names. Includes the `SDKPaths` patch (singleton
`is_release`/dist-packages bug hit repeatedly on non-release installs).

In [ ]:
import tempfile as _tempfile

from hailo_sdk_common.paths_manager.paths import SDKPaths

_p = SDKPaths()
if not _p.is_release:
    _p._is_release = True
    _p._build_dir = _tempfile.mkdtemp(prefix=type(_p).HAILO_TEMP_DIR_PREFIX)
print(f"SDKPaths patch applied: is_release={_p.is_release}")

# Diagnostics-only option: also expose the unduplicated base scope as a
# third network group (written to model_basescope.hef). That is what
# runtime/diagnostics/generate_base_scope.py drives for cache-free greedy
# generation (full-prefix recomputation). WARNING: three networks on-chip
# instead of two tends to break hailo-ollama / genai.LLM() -- the official
# Qwen compile recipe declares exactly two groups. Keep False for a HEF
# meant to be deployed through the genai stack.
INCLUDE_BASE_SCOPE = False

base_group_line = (
    f"{NET_SCOPE} = network_group([{NET_SCOPE}])" if INCLUDE_BASE_SCOPE else ""
)
compile_script = f"""
performance_param(compiler_optimization_level=0)
{NET_SCOPE}__prefill = network_group([{NET_SCOPE}__prefill])
{NET_SCOPE}__tbt = network_group([{NET_SCOPE}__tbt])
{base_group_line}
"""
print("=== compile script ===")
print(compile_script)

runner = ClientRunner(har=CONVFIXED_HAR_PATH)
runner.load_model_script(compile_script)
hef_bytes = runner.compile()

out_path = os.path.join(WORKDIR, "model_basescope.hef") if INCLUDE_BASE_SCOPE else HEF_PATH
with open(out_path, "wb") as f:
    f.write(hef_bytes)
print(f"\nHEF written: {out_path} ({len(hef_bytes) / 1024 / 1024:.2f} MiB)")
if INCLUDE_BASE_SCOPE:
    print("[OK] step 8 validated -- diagnostics-only base-scope HEF (not for genai serving)")
else:
    print("[OK] step 8 validated -- compile succeeded (minutes; monolithic lm_head included)")

## Deploy & serve (device host)

Copy the HEF to the machine that has the Hailo-10H, register it into
hailo-ollama's model store, restart the server (manifests are scanned at
startup only), and generate:

```bash
scp workdir/model.hef device-host:/tmp/

python runtime/register_hailo_ollama.py --hef /tmp/model.hef \
    --family tinystories25m --tag nb-v1

# restart hailo-ollama serve, then:
curl -s -H 'Content-Type: application/json' http://localhost:8000/api/generate \
  -d '{"model":"tinystories25m:nb-v1","prompt":"Once upon a time","stream":false}'
```

**Expected today:** coherent greedy generation inside the base scope;
prefill numerically exact (cosine ≈ 1.0 vs float32 reference); degraded
multi-token coherence through `__tbt` — the
[open issue](../docs/findings/open-tbt-cache-read.md).

Layer-by-layer validation instruments (wire encodings, static HEF audit,
manual prefill/tbt probes) live in the
[diagnostics notebook](diagnostics.ipynb); the full investigation trail is
in [docs/findings/](../docs/findings/index.md).